# Recursive Local Model

Generates and runs code to operate on the context.

In [ ]:
#%pip install --quiet rlms 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../keys.env")
assert os.environ["OPENAI_API_KEY"][:2] == "sk",\
       "Please specify the OPEN_API_KEY access token in keys.env file"

In [2]:
import sys
sys.path.append('../06_basic_rag')
import gutenberg_text_loader as gtl
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

The examples here on the book Anabasis of Alexander https://www.gutenberg.org/cache/epub/46976/pg46976.txt
a 2nd century historical account of Alexander the Great

In [4]:
text = gtl.GutenbergSource().load_cleaned_text("https://www.gutenberg.org/cache/epub/46976/pg46976.txt")

2026-08-02 14:51:18,988 - INFO - Created cache directory at .cache
2026-08-02 14:51:18,989 - INFO - Fetching text from URL: https://www.gutenberg.org/cache/epub/46976/pg46976.txt
2026-08-02 14:51:19,324 - INFO - Cached content for https://www.gutenberg.org/cache/epub/46976/pg46976.txt at .cache/pg46976_1b16d8525c.txt
2026-08-02 14:51:19,335 - INFO - Cleaned Gutenberg text: removed 1203 chars from start, 18808 chars from end


In [6]:
from rlm import RLM

rlm = RLM(
    backend="openai",
    backend_kwargs={"model_name": "gpt-5-nano"},
    verbose=True,  # For printing to console with rich, disabled by default.
)

╭─ ◆ RLM ━ Recursive Language Model ──────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    Backend                            openai                   Environment                       local          │
│    Model                              gpt-5-nano               Max Iterations                    30             │
│    Max Depth                          1                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [11]:
response = rlm.completion(f"""
Read the following book and tell me how many battles Alexander fought. 
List the battles and tell me who was it against and where the battle took place. 

{text}""")

2026-08-02 15:03:30,363 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


──────────────────────────────────────────────────  Iteration 1  ──────────────────────────────────────────────────

╭─ ◇ LLM Response  (11.66s) ──────────────────────────────────────────────────────────────────────────────────────╮
│ ```repl                                                                                                         │
│ print("Context probe:")                                                                                         │
│ print("Total length:", len(context))                                                                            │
│ print("Head (first 2000 chars):")                                                                               │
│ print(context[:2000])                                                                                           │
│ print("Tail (last 2000 chars):")                                                                                │
│ print(context[-2000:])                                                                                          │
│ print("Line count (approx):", context.count("\n") + 1)                                                          │
│ ```                                                                                                             │
│                                                                                                                 │
│ ~23 words                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ▸ Code Execution  (0.002s) ────────────────────────────────────────────────────────────────────────────────────╮
│ Code:                                                                                                           │
│ print("Context probe:")                                                                                         │
│ print("Total length:", len(context))                                                                            │
│ print("Head (first 2000 chars):")                                                                               │
│ print(context[:2000])                                                                                           │
│ print("Tail (last 2000 chars):")                                                                                │
│ print(context[-2000:])                                                                                          │
│ print("Line count (approx):", context.count("\n") + 1)                                                          │
│                                                                                                                 │
│ Output:                                                                                                         │
│ Context probe:                                                                                                  │
│ Total length: 877027                                                                                            │
│ Head (first 2000 chars):                                                                                        │
│                                                                                                                 │
│ Read the following book and tell me how many battles Alexander fought.                                          │
│ List the battles and tell me who was it against and where the battle took place.                                │
│                                                                                                                 │
│ THE                                                                                                             │
│                                                                                                                 │
│   ANABASIS OF ALEXANDER.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│   THE                                                                                                           │
│                                                                                                                 │
│   ANABASIS OF ALEXANDER;                                                                                        │
│                                                                                                                 │
│   OR,                                                                                                           │
│                                                                                                                 │
│   The History of the Wars and Conquests of Alexander the Great.                                                 │
│                                                                                                                 │
│   _LITERALLY TRANSLATED, WITH A COMMENTARY,                                                                     │
│   FROM THE GREEK OF ARRIAN THE NICOMEDIAN_,           

2026-08-02 15:04:02,084 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-02 15:04:34,005 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

╭─ ▸ Code Execution  (0.002s) ────────────────────────────────────────────────────────────────────────────────────╮
│ Code:                                                                                                           │
│ # Turn 30 plan and extraction for chunk 28 (29th slice), then merge with existing merged_battles (chunks 0–28)  │
│                                                                                                                 │
│ print("Turn 30 Plan:")                                                                                          │
│ print("- Extract battles from chunk 28 (29th slice).")                                                          │
│ print("- Merge with existing merged_battles (chunks 0–28), deduplicating overlaps while preserving order.")     │
│ print("- Report cumulative count and show a preview of the combined list.")                                     │
│ print("Proceeding to extract from chunk 28 now...")                                                             │
│                                                                                                                 │
│ # Identify chunk 28 (29th slice)                                                                                │
│ chunk28 = chunks[28] if len(chunks) > 28 else ""                                                                │
│ print("Chunk 28 length (chars):", len(chunk28))                                                                 │
│                                                                                                                 │
│ # Build prompt for chunk 28                                                                                     │
│ if chunk28:                                                                                                     │
│     prompt28 = f"""                                                                                             │
│ From the following text chunk about Alexander the Great (as described in Arrian's The Anabasis),                │
│ extract every battle that is named or described. Return a JSON array of objects with the fields:                │
│ - name: battle name (string)                                                                                    │
│ - opponents: opponents (string or array)                                                                        │
│ - location: where the battle took place (string)                                                                │
│ - year: year if given (string, optional)                                                                        │
│ If a field is not available, use an empty string or null.                                                       │
│                                                                                                                 │
│ Text chunk:                                                                                                     │
│ \"\"\"{chunk28}\"\"\"                                                                                           │
│ """                                                                                                             │
│     # Sub-call to the LLM for chunk 28                                                                          │
│     resp28 = llm_query(prompt28, model=None)                                                                    │
│                                                                                                                 │
│     def extract_json_list(text):                                                                                │
│         m = __import__("re").search(r'\[.*\]', text, __import__("re").S)                                        │
│         if not m:                                                                                               │
│             return []                                 

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Turn 30 complete. Total battles: 62. Preview: [{'name': 'Battle with the Triballians', 'opponents':            │
│  'Triballians', 'location': 'Lyginus River', 'year': 'B.C. 335'}, {'name': 'Defeat of Clitus and Glaucias',     │
│  'opponents': ['Clitus', 'Glaucias'], 'location': 'Pelium region (Erigon area)', 'year': 'B.C. 335'}, {'name':  │
│  'Revolt of Thebes', 'opponents': 'Thebans', 'location': 'Thebes', 'year': 'B.C. 335'}, {'name': 'Fall of       │
│  Thebes', 'opponents': 'Thebans', 'location': 'Thebes', 'year': 'B.C. 335'}, {'name': 'Destruction of Thebes',  │
│  'opponents': 'Thebans', 'location': 'Thebes', 'year': 'B.C. 335'}]                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       30                                                   
                                             Total Time       982.25s                                              
                                             Input Tokens     825,278                                              
                                             Output Tokens    121,328  

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

In [16]:
import textwrap
print(textwrap.fill(response.response, width=70))

Turn 30 complete. Total battles: 62. Preview: [{'name': 'Battle with
the Triballians', 'opponents': 'Triballians', 'location': 'Lyginus
River', 'year': 'B.C. 335'}, {'name': 'Defeat of Clitus and Glaucias',
'opponents': ['Clitus', 'Glaucias'], 'location': 'Pelium region
(Erigon area)', 'year': 'B.C. 335'}, {'name': 'Revolt of Thebes',
'opponents': 'Thebans', 'location': 'Thebes', 'year': 'B.C. 335'},
{'name': 'Fall of Thebes', 'opponents': 'Thebans', 'location':
'Thebes', 'year': 'B.C. 335'}, {'name': 'Destruction of Thebes',
'opponents': 'Thebans', 'location': 'Thebes', 'year': 'B.C. 335'}]
